# 📘 Lakeflow Spark Declarative Pipelines (formerly Delta Live Tables)
## Databricks Notebook — Declarative Data Pipelines

**2025 Naming Update:** Delta Live Tables (DLT) is now called **Lakeflow Spark Declarative Pipelines**.
The `import dlt` module still works; `import pipelines` is the new standard. No code migration required.

**What you'll learn:**
- DLT/SDP concepts: declarative vs imperative pipelines
- Syntax: `@dlt.table`, `@dlt.view`, expectations (quality rules)
- Auto Loader for incremental file ingestion
- Streaming tables vs materialized views
- Change Data Feed (CDF)
- NEW IoT/Manufacturing dataset (200K+ sensor readings)

**Note:** DLT code runs inside pipeline context, not regular notebooks.
This notebook shows DLT code (conceptual) alongside runnable equivalents for Community Edition.

**Prerequisites:** Notebooks 01-05

---

---
# 🏗️ IoT/Manufacturing Dataset Creation

In [ ]:
%sql
CREATE DATABASE IF NOT EXISTS iot_manufacturing_dw;
USE iot_manufacturing_dw;

In [ ]:
%sql
-- Machines (30 rows)
DROP TABLE IF EXISTS machines;
CREATE TABLE machines (
  machine_id INT, machine_name STRING, machine_type STRING, production_line STRING,
  install_date DATE, maintenance_schedule STRING, status STRING
);
INSERT INTO machines
SELECT id AS machine_id,
  CONCAT('Machine_', LPAD(CAST(id AS STRING),3,'0')) AS machine_name,
  CASE abs(hash(id*3))%5 WHEN 0 THEN 'CNC_Mill' WHEN 1 THEN 'Lathe' WHEN 2 THEN 'Press' WHEN 3 THEN 'Robot_Arm' ELSE 'Conveyor' END AS machine_type,
  CONCAT('Line_', CAST(abs(hash(id*7))%5+1 AS STRING)) AS production_line,
  date_add('2018-01-01', abs(hash(id*11))%2000) AS install_date,
  CASE abs(hash(id*13))%3 WHEN 0 THEN 'weekly' WHEN 1 THEN 'monthly' ELSE 'quarterly' END AS maintenance_schedule,
  CASE abs(hash(id*17))%4 WHEN 0 THEN 'running' WHEN 1 THEN 'running' WHEN 2 THEN 'idle' ELSE 'maintenance' END AS status
FROM (SELECT explode(sequence(1, 30)) AS id);

In [ ]:
%sql
-- Sensors (100 rows)
DROP TABLE IF EXISTS sensors;
CREATE TABLE sensors (
  sensor_id INT, sensor_type STRING, location STRING, machine_id INT,
  install_date DATE, calibration_date DATE, status STRING
);
INSERT INTO sensors
SELECT id AS sensor_id,
  CASE abs(hash(id*3))%4 WHEN 0 THEN 'temperature' WHEN 1 THEN 'pressure' WHEN 2 THEN 'vibration' ELSE 'humidity' END AS sensor_type,
  CONCAT('Zone_', CAST(abs(hash(id*7))%10+1 AS STRING)) AS location,
  abs(hash(id*11))%30+1 AS machine_id,
  date_add('2020-01-01', abs(hash(id*13))%1000) AS install_date,
  date_add('2023-01-01', abs(hash(id*17))%365) AS calibration_date,
  CASE abs(hash(id*19))%5 WHEN 0 THEN 'active' WHEN 1 THEN 'active' WHEN 2 THEN 'active' WHEN 3 THEN 'needs_calibration' ELSE 'offline' END AS status
FROM (SELECT explode(sequence(1, 100)) AS id);

In [ ]:
%sql
-- Sensor Readings (200000 rows) - high volume time series
DROP TABLE IF EXISTS sensor_readings;
CREATE TABLE sensor_readings (
  reading_id INT, sensor_id INT, reading_timestamp TIMESTAMP,
  value DECIMAL(10,4), unit STRING, quality_flag STRING, is_anomaly BOOLEAN
);
INSERT INTO sensor_readings
SELECT id AS reading_id,
  abs(hash(id*3))%100+1 AS sensor_id,
  CAST(date_add('2024-01-01', abs(hash(id*7))%180) AS TIMESTAMP) AS reading_timestamp,
  CAST(abs(hash(id*11))%10000/100.0 AS DECIMAL(10,4)) AS value,
  CASE abs(hash(id*3))%4 WHEN 0 THEN 'celsius' WHEN 1 THEN 'psi' WHEN 2 THEN 'mm/s' ELSE 'percent' END AS unit,
  CASE abs(hash(id*13))%10 WHEN 0 THEN 'suspect' WHEN 1 THEN 'invalid' ELSE 'valid' END AS quality_flag,
  CASE WHEN abs(hash(id*17))%50=0 THEN true ELSE false END AS is_anomaly
FROM (SELECT explode(sequence(1, 200000)) AS id);

In [ ]:
%sql
-- Production Events (50000 rows)
DROP TABLE IF EXISTS production_events;
CREATE TABLE production_events (
  event_id INT, machine_id INT, event_type STRING, event_timestamp TIMESTAMP,
  duration_minutes INT, operator_id INT, notes STRING
);
INSERT INTO production_events
SELECT id AS event_id,
  abs(hash(id*3))%30+1 AS machine_id,
  CASE abs(hash(id*7))%4 WHEN 0 THEN 'start' WHEN 1 THEN 'stop' WHEN 2 THEN 'error' ELSE 'maintenance' END AS event_type,
  CAST(date_add('2024-01-01', abs(hash(id*11))%180) AS TIMESTAMP) AS event_timestamp,
  abs(hash(id*13))%480+1 AS duration_minutes,
  abs(hash(id*17))%20+1 AS operator_id,
  CASE WHEN abs(hash(id*19))%5=0 THEN 'Requires follow-up' ELSE NULL END AS notes
FROM (SELECT explode(sequence(1, 50000)) AS id);

In [ ]:
%sql
-- Quality Checks (20000 rows)
DROP TABLE IF EXISTS quality_checks;
CREATE TABLE quality_checks (
  check_id INT, production_batch_id INT, machine_id INT, check_timestamp TIMESTAMP,
  measurement DECIMAL(8,4), tolerance_min DECIMAL(8,4), tolerance_max DECIMAL(8,4),
  pass_fail STRING, inspector_id INT
);
INSERT INTO quality_checks
SELECT id AS check_id,
  abs(hash(id*3))%1000+1 AS production_batch_id,
  abs(hash(id*7))%30+1 AS machine_id,
  CAST(date_add('2024-01-01', abs(hash(id*11))%180) AS TIMESTAMP) AS check_timestamp,
  CAST(abs(hash(id*13))%1000/100.0 AS DECIMAL(8,4)) AS measurement,
  CAST(2.0 AS DECIMAL(8,4)) AS tolerance_min,
  CAST(8.0 AS DECIMAL(8,4)) AS tolerance_max,
  CASE WHEN abs(hash(id*13))%1000/100.0 BETWEEN 2.0 AND 8.0 THEN 'pass' ELSE 'fail' END AS pass_fail,
  abs(hash(id*17))%10+1 AS inspector_id
FROM (SELECT explode(sequence(1, 20000)) AS id);

In [ ]:
%sql
SELECT 'machines' AS tbl, COUNT(*) AS rows FROM machines
UNION ALL SELECT 'sensors', COUNT(*) FROM sensors
UNION ALL SELECT 'sensor_readings', COUNT(*) FROM sensor_readings
UNION ALL SELECT 'production_events', COUNT(*) FROM production_events
UNION ALL SELECT 'quality_checks', COUNT(*) FROM quality_checks
ORDER BY tbl;

---
# 📗 Section 1: DLT / Declarative Pipeline Concepts

## What Is It?

**Lakeflow Spark Declarative Pipelines** (formerly Delta Live Tables) lets you define
*what* your data should look like, not *how* to compute it. The engine handles:
- Execution order (dependency resolution)
- Incremental processing
- Error handling and retries
- Data quality enforcement
- Schema management

## Declarative vs Imperative

```
IMPERATIVE (traditional):              DECLARATIVE (DLT/SDP):
─────────────────────────              ────────────────────────
df = spark.read(...)                   @dlt.table
df2 = df.filter(...)                   def silver_orders():
df2.write.saveAsTable(...)               return dlt.read("bronze_orders")
                                                .filter(...)
# YOU manage:                          # ENGINE manages:
# - execution order                    # - execution order
# - error handling                     # - error handling
# - incremental logic                  # - incremental logic
# - schema evolution                   # - schema evolution
```

## When to Use DLT vs Regular Notebooks

| Use DLT When | Use Regular Notebooks When |
|---|---|
| Multi-table ETL pipelines | Ad-hoc analysis |
| Need built-in quality checks | Complex ML workflows |
| Want automatic incremental | One-off data fixes |
| Production data pipelines | Exploratory work |
| Team collaboration on ETL | Custom scheduling needs |

## DLT vs Traditional Spark Pipelines

### The Problem with Traditional Pipelines

```python
# Traditional Spark pipeline — you manage EVERYTHING:
def run_pipeline():
    # 1. You handle dependencies manually
    raw = spark.read.format("cloudFiles").load("/raw/orders/")
    
    # 2. You write quality checks yourself (or forget them)
    if raw.filter(col("order_id").isNull()).count() > 0:
        raise ValueError("NULL order_ids found!")
    
    # 3. You manage incremental state yourself
    last_run = get_last_watermark()
    new_data = raw.filter(col("ts") > last_run)
    
    # 4. You handle retries, checkpoints, monitoring yourself
    new_data.write.format("delta").mode("append").saveAsTable("silver.orders")
    
    # 5. You update the watermark yourself
    update_watermark(new_data.agg(max("ts")).collect()[0][0])
```

### With DLT — Declarative, Not Imperative

```python
# DLT pipeline — you declare WHAT, Databricks handles HOW:
import dlt
from pyspark.sql.functions import col

@dlt.table(comment="Raw orders from Auto Loader")
def raw_orders():
    return (spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .load("/raw/orders/"))

@dlt.table(comment="Validated orders — Silver layer")
@dlt.expect_or_drop("valid_order_id", "order_id IS NOT NULL")
@dlt.expect_or_drop("positive_amount", "total_amount > 0")
def silver_orders():
    return (dlt.read_stream("raw_orders")
        .withColumn("status", lower(trim(col("status")))))

@dlt.table(comment="Daily revenue — Gold layer")
def gold_daily_revenue():
    return (dlt.read("silver_orders")
        .groupBy("order_date")
        .agg(sum("total_amount").alias("revenue")))
```

### What DLT Handles Automatically

| Concern | Traditional | DLT |
|---------|-------------|-----|
| **Dependency ordering** | You manage | Auto-detected from `dlt.read()` |
| **Incremental state** | Manual watermarks | Built-in checkpointing |
| **Quality checks** | Custom code | `@dlt.expect` decorators |
| **Retries** | Manual retry logic | Automatic with backoff |
| **Monitoring** | Custom dashboards | Built-in pipeline UI |
| **Schema evolution** | Manual handling | Automatic |
| **Backfill** | Complex reprocessing | `FULL REFRESH` command |
| **Lineage** | Not tracked | Auto-tracked in Unity Catalog |

In [ ]:
# DLT concepts simulation (since DLT requires a pipeline context)
print("📋 LAKEFLOW DECLARATIVE PIPELINES — Key Concepts")
print("=" * 60)

# Simulate what DLT does behind the scenes
class DLTSimulator:
    """Simulates how DLT manages pipeline execution."""
    
    def __init__(self, pipeline_name):
        self.name = pipeline_name
        self.tables = {}
        self.dependencies = {}
        self.expectations = {}
        self.execution_order = []
    
    def table(self, name, depends_on=None, expectations=None):
        """Register a DLT table (like @dlt.table decorator)."""
        self.tables[name] = {"status": "pending", "rows": 0}
        self.dependencies[name] = depends_on or []
        self.expectations[name] = expectations or []
    
    def _get_execution_order(self):
        """Topological sort — DLT auto-detects this from dlt.read() calls."""
        visited = set()
        order = []
        
        def visit(name):
            if name in visited:
                return
            visited.add(name)
            for dep in self.dependencies.get(name, []):
                visit(dep)
            order.append(name)
        
        for name in self.tables:
            visit(name)
        return order
    
    def run(self, mode="incremental"):
        """Execute the pipeline."""
        print(f"\n🚀 Running pipeline: {self.name} (mode={mode})")
        order = self._get_execution_order()
        print(f"   Execution order (auto-detected): {' → '.join(order)}")
        
        for table_name in order:
            deps = self.dependencies[table_name]
            exps = self.expectations[table_name]
            
            # Simulate execution
            self.tables[table_name]["status"] = "running"
            
            # Check expectations
            violations = 0
            for exp_name, exp_action in exps:
                # Simulate 2% violation rate
                import random
                if random.random() < 0.02:
                    violations += 1
                    if exp_action == "fail":
                        self.tables[table_name]["status"] = "failed"
                        print(f"   ❌ {table_name}: FAILED — expectation '{exp_name}' violated")
                        return
            
            self.tables[table_name]["status"] = "completed"
            self.tables[table_name]["rows"] = random.randint(1000, 50000)
            
            status = "✅"
            print(f"   {status} {table_name}: {self.tables[table_name]['rows']:,} rows"
                  f"{f' ({violations} expectation violations)' if violations else ''}")
    
    def show_lineage(self):
        """Show the data lineage graph."""
        print(f"\n📊 Pipeline Lineage:")
        for table, deps in self.dependencies.items():
            if deps:
                print(f"   {' + '.join(deps)} → {table}")
            else:
                print(f"   [source] → {table}")


# Build a DLT pipeline
import random
random.seed(42)

pipeline = DLTSimulator("techmart_medallion_pipeline")

# Register tables (like @dlt.table decorators)
pipeline.table("raw_orders")  # Source (Auto Loader)
pipeline.table("raw_customers")  # Source
pipeline.table("silver_orders", 
               depends_on=["raw_orders"],
               expectations=[("valid_order_id", "drop"), ("positive_amount", "drop")])
pipeline.table("silver_customers",
               depends_on=["raw_customers"],
               expectations=[("valid_email", "warn")])
pipeline.table("gold_daily_revenue",
               depends_on=["silver_orders"])
pipeline.table("gold_customer_360",
               depends_on=["silver_orders", "silver_customers"])

pipeline.show_lineage()
pipeline.run()


---
# 📗 Section 2: DLT Syntax & Decorators

## Legacy Syntax (still works) vs New Syntax (2025+)

⚠️ **These cells are CONCEPTUAL** — they show DLT code that runs inside a pipeline context,
not in a regular notebook. Runnable equivalents follow in Section 6.

In [ ]:
# ⚠️ CONCEPTUAL — runs inside DLT pipeline only
# ============================================
# LEGACY SYNTAX (import dlt)
# ============================================

# import dlt
#
# @dlt.table(
#     comment="Raw sensor readings from IoT devices",
#     table_properties={"quality": "bronze"}
# )
# def bronze_sensor_readings():
#     return spark.table("iot_manufacturing_dw.sensor_readings")
#
# @dlt.table(comment="Cleaned sensor readings")
# def silver_sensor_readings():
#     return (
#         dlt.read("bronze_sensor_readings")
#         .filter("quality_flag = 'valid'")
#         .filter("value IS NOT NULL")
#     )

print("⚠️ DLT code above is conceptual — runs inside pipeline context only")
print("See Section 6 for runnable equivalents")

In [ ]:
# ⚠️ CONCEPTUAL — runs inside DLT pipeline only
# ============================================
# NEW SYNTAX (import pipelines) — 2025+
# ============================================

# import pipelines
#
# @pipelines.table(
#     comment="Raw sensor readings from IoT devices",
#     table_properties={"quality": "bronze"}
# )
# def bronze_sensor_readings():
#     return spark.table("iot_manufacturing_dw.sensor_readings")
#
# @pipelines.table(comment="Cleaned sensor readings")
# def silver_sensor_readings():
#     return (
#         pipelines.read("bronze_sensor_readings")
#         .filter("quality_flag = 'valid'")
#         .filter("value IS NOT NULL")
#     )

print("⚠️ New 'pipelines' syntax is equivalent to 'dlt' — same behavior")
print("Both syntaxes will continue to work")

## DLT Syntax Reference — Complete Guide

### Table Types

| Type | Decorator | Storage | Incremental? | Use For |
|------|-----------|---------|-------------|---------|
| **Streaming Table** | `@dlt.table` + `readStream` | Delta | ✅ Yes | Append-only sources (Kafka, Auto Loader) |
| **Materialized View** | `@dlt.table` + `read` | Delta | ✅ Yes (smart) | Aggregations, joins |
| **View** | `@dlt.view` | None (virtual) | N/A | Intermediate transformations |

### Complete DLT Syntax Examples

```python
import dlt
from pyspark.sql.functions import col, lower, trim, sum, count, current_timestamp

# ─────────────────────────────────────────────────────────────────
# BRONZE: Streaming Table from Auto Loader
# ─────────────────────────────────────────────────────────────────
@dlt.table(
    name="bronze_orders",
    comment="Raw orders from S3 via Auto Loader",
    table_properties={
        "quality": "bronze",
        "pipelines.autoOptimize.managed": "true"
    }
)
def bronze_orders():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", "/checkpoints/orders/schema")
        .option("cloudFiles.inferColumnTypes", "true")
        .load("/raw/orders/")
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source", lit("s3://raw/orders/"))
    )

# ─────────────────────────────────────────────────────────────────
# SILVER: Streaming Table with Expectations
# ─────────────────────────────────────────────────────────────────
@dlt.table(
    name="silver_orders",
    comment="Cleaned and validated orders",
    table_properties={"quality": "silver"}
)
@dlt.expect("valid_order_id", "order_id IS NOT NULL")
@dlt.expect_or_drop("positive_amount", "total_amount > 0")
@dlt.expect_or_fail("no_future_dates", "order_date <= current_date()")
def silver_orders():
    return (
        dlt.read_stream("bronze_orders")
        .withColumn("status", lower(trim(col("status"))))
        .withColumn("total_amount", col("total_amount").cast("decimal(12,2)"))
        .filter(col("customer_id").isNotNull())
    )

# ─────────────────────────────────────────────────────────────────
# GOLD: Materialized View (aggregation)
# ─────────────────────────────────────────────────────────────────
@dlt.table(
    name="gold_daily_revenue",
    comment="Daily revenue metrics for Finance dashboard",
    table_properties={"quality": "gold"}
)
def gold_daily_revenue():
    return (
        dlt.read("silver_orders")
        .groupBy("order_date")
        .agg(
            count("*").alias("total_orders"),
            sum("total_amount").alias("total_revenue")
        )
    )
```

### Expectations — Data Quality in DLT

| Decorator | On Violation | Use When |
|-----------|-------------|----------|
| `@dlt.expect` | Log warning, keep row | Soft check (informational) |
| `@dlt.expect_or_drop` | Drop the row | Remove bad records silently |
| `@dlt.expect_or_fail` | Fail the pipeline | Critical data must be valid |
| `@dlt.expect_all` | Log all violations | Multiple checks, keep rows |
| `@dlt.expect_all_or_drop` | Drop if ANY fails | Multiple checks, drop bad |
| `@dlt.expect_all_or_fail` | Fail if ANY fails | Critical multi-check |

In [ ]:
# Expectations simulation — understand the behavior
print("🔍 DLT EXPECTATIONS — Behavior Comparison")
print("=" * 60)

# Simulate records with quality issues
records = [
    {"order_id": 1, "amount": 100, "date": "2024-03-15"},
    {"order_id": None, "amount": 50, "date": "2024-03-15"},   # NULL order_id
    {"order_id": 3, "amount": -10, "date": "2024-03-15"},     # Negative amount
    {"order_id": 4, "amount": 200, "date": "2099-01-01"},     # Future date
    {"order_id": 5, "amount": 300, "date": "2024-03-16"},
]

from datetime import date

def check_expectations(records, expectations):
    """Simulate DLT expectation behavior."""
    results = {"kept": [], "dropped": [], "violations": []}
    pipeline_failed = False
    
    for record in records:
        record_violations = []
        should_drop = False
        
        for exp_name, check_fn, action in expectations:
            if not check_fn(record):
                record_violations.append(exp_name)
                if action == "drop":
                    should_drop = True
                elif action == "fail":
                    pipeline_failed = True
                    print(f"   ❌ PIPELINE FAILED: {exp_name} violated by {record}")
                    return results, True
        
        if record_violations:
            results["violations"].append({"record": record, "violations": record_violations})
        
        if should_drop:
            results["dropped"].append(record)
        else:
            results["kept"].append(record)
    
    return results, pipeline_failed


# Define expectations
expectations = [
    ("valid_order_id", lambda r: r["order_id"] is not None, "warn"),
    ("positive_amount", lambda r: r["amount"] > 0, "drop"),
    ("no_future_dates", lambda r: r["date"] <= "2024-12-31", "warn"),
]

print("\nInput records:", len(records))
results, failed = check_expectations(records, expectations)

print(f"\n📊 Results:")
print(f"   ✅ Kept: {len(results['kept'])} records")
print(f"   ❌ Dropped: {len(results['dropped'])} records")
print(f"   ⚠️ Violations logged: {len(results['violations'])}")

print(f"\n   Dropped records (expect_or_drop):")
for r in results["dropped"]:
    print(f"      {r}")

print(f"\n   Violations (expect — logged but kept):")
for v in results["violations"]:
    if v["record"] not in results["dropped"]:
        print(f"      {v['record']} → {v['violations']}")


---
# 📗 Section 3: Expectations (Data Quality Rules)

## Three Violation Modes

| Decorator | On Violation | Use Case |
|---|---|---|
| `@dlt.expect("name", "condition")` | Warn (log metric) | Non-critical, monitor |
| `@dlt.expect_or_drop("name", "cond")` | Drop bad rows | Filter invalid data |
| `@dlt.expect_or_fail("name", "cond")` | Fail pipeline | Critical data integrity |

⚠️ Conceptual DLT code below — runnable equivalent follows.

In [ ]:
# ⚠️ CONCEPTUAL — DLT Expectations syntax
# ============================================
# @dlt.table
# @dlt.expect("valid_reading", "value IS NOT NULL")
# @dlt.expect("positive_value", "value >= 0")
# @dlt.expect_or_drop("valid_sensor", "sensor_id IS NOT NULL")
# @dlt.expect_or_fail("valid_timestamp", "reading_timestamp IS NOT NULL")
# def silver_sensor_readings():
#     return dlt.read("bronze_sensor_readings")

# RUNNABLE EQUIVALENT: Implement expectations as filters + metrics
from pyspark.sql.functions import *

readings = spark.table("iot_manufacturing_dw.sensor_readings")
total = readings.count()

# Check each expectation
checks = {
    "valid_reading (value IS NOT NULL)": readings.filter("value IS NOT NULL").count(),
    "positive_value (value >= 0)": readings.filter("value >= 0").count(),
    "valid_sensor (sensor_id IS NOT NULL)": readings.filter("sensor_id IS NOT NULL").count(),
    "valid_timestamp": readings.filter("reading_timestamp IS NOT NULL").count(),
}

print(f"Total readings: {total:,}")
print(f"\nExpectation Results:")
for name, passing in checks.items():
    pct = passing / total * 100
    status = "✅" if pct >= 99 else "⚠️" if pct >= 95 else "❌"
    print(f"  {status} {name}: {passing:,}/{total:,} ({pct:.1f}%)")

In [ ]:
# Runnable: Apply expect_or_drop equivalent
silver_readings = (readings
    .filter("value IS NOT NULL")          # expect_or_drop
    .filter("value >= 0")                 # expect_or_drop
    .filter("sensor_id IS NOT NULL")      # expect_or_drop
    .filter("reading_timestamp IS NOT NULL")  # expect_or_fail equivalent
    .filter("quality_flag = 'valid'")     # business rule
)

dropped = total - silver_readings.count()
print(f"After quality filters: {silver_readings.count():,} rows ({dropped:,} dropped)")

In [ ]:
# ============================================
# 🎯 YOUR TURN: Define Expectations
# ============================================
# For the quality_checks table, define and apply expectations:
# 1. check_id IS NOT NULL (fail if violated)
# 2. measurement IS NOT NULL (drop if violated)
# 3. measurement BETWEEN tolerance_min AND tolerance_max (warn only)
# 4. pass_fail IN ('pass', 'fail') (drop if violated)
#
# Print the pass rate for each expectation
# Apply the drop rules and show final count
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
from pyspark.sql.functions import *

qc = spark.table("iot_manufacturing_dw.quality_checks")
total = qc.count()

expectations = {
    "check_id NOT NULL (fail)": qc.filter("check_id IS NOT NULL").count(),
    "measurement NOT NULL (drop)": qc.filter("measurement IS NOT NULL").count(),
    "in_tolerance (warn)": qc.filter("measurement BETWEEN tolerance_min AND tolerance_max").count(),
    "valid_pass_fail (drop)": qc.filter("pass_fail IN ('pass', 'fail')").count(),
}

print(f"Quality Checks: {total:,} total")
for name, passing in expectations.items():
    pct = passing / total * 100
    mode = "FAIL" if "fail" in name else "DROP" if "drop" in name else "WARN"
    print(f"  [{mode}] {name}: {pct:.1f}%")

# Apply drop rules
silver_qc = (qc
    .filter("check_id IS NOT NULL")
    .filter("measurement IS NOT NULL")
    .filter("pass_fail IN ('pass', 'fail')")
)
print(f"\nAfter drops: {silver_qc.count():,} rows")

---
# 📗 Section 4: Auto Loader

## What Is Auto Loader?

Auto Loader (`cloudFiles`) incrementally ingests new files as they arrive in cloud storage.
It tracks which files have been processed using a checkpoint.

```
Cloud Storage (S3/ADLS/GCS)          Auto Loader           Delta Table
┌─────────────────────────┐    ┌──────────────────┐    ┌──────────────┐
│ file_001.json ✓         │    │ Checkpoint:      │    │ Bronze Table │
│ file_002.json ✓         │───▶│ - file_001 ✓     │───▶│ (append)     │
│ file_003.json (NEW)     │    │ - file_002 ✓     │    │              │
│ file_004.json (NEW)     │    │ - file_003 (new) │    │              │
└─────────────────────────┘    └──────────────────┘    └──────────────┘
```

**Key features:**
- Exactly-once processing (no duplicates)
- Schema inference and evolution
- Supports JSON, CSV, Parquet, Avro, text
- Scales to millions of files

Reference: https://docs.databricks.com/ingestion/cloud-object-storage/auto-loader/

In [ ]:
# ⚠️ CONCEPTUAL — Auto Loader syntax (requires cloud storage)
# ============================================
# In DLT pipeline:
#
# @dlt.table
# def bronze_sensor_readings():
#     return (
#         spark.readStream
#         .format("cloudFiles")
#         .option("cloudFiles.format", "json")
#         .option("cloudFiles.schemaLocation", "/checkpoints/sensors/schema")
#         .option("cloudFiles.inferColumnTypes", "true")
#         .load("/mnt/iot-data/sensors/")
#     )
#
# In regular notebook (Structured Streaming):
#
# df = (spark.readStream
#     .format("cloudFiles")
#     .option("cloudFiles.format", "json")
#     .option("cloudFiles.schemaLocation", "/tmp/schema")
#     .load("/mnt/data/incoming/")
# )
# df.writeStream.format("delta").option("checkpointLocation", "/tmp/cp").toTable("bronze")

print("Auto Loader requires cloud storage (S3/ADLS/GCS)")
print("Not available on Community Edition local storage")
print("See documentation: https://docs.databricks.com/ingestion/cloud-object-storage/auto-loader/")

---
# 📗 Section 5: Streaming Tables vs Materialized Views

## Key Differences

| Feature | Streaming Table | Materialized View |
|---|---|---|
| Processing | Append-only, incremental | Full recompute |
| Source | Append-only streams | Any table (incl. updates) |
| Use case | Raw ingestion, logs, events | Aggregations, joins |
| Supports DELETE/UPDATE | ❌ No | ✅ Yes |
| Performance | Very fast (only new data) | Slower (full recompute) |

## When to Use Each

```
Streaming Table:                    Materialized View:
─────────────────                   ──────────────────
• Bronze layer (raw ingestion)      • Silver layer (cleaned)
• Event logs                        • Gold layer (aggregates)
• Sensor readings                   • Dimension tables
• Clickstream data                  • KPI tables
• Append-only sources               • Sources with updates
```

In [ ]:
# ⚠️ CONCEPTUAL — DLT streaming table vs materialized view
#
# STREAMING TABLE (append-only, incremental):
# @dlt.table
# def bronze_readings():
#     return spark.readStream.table("raw_sensor_data")
#
# MATERIALIZED VIEW (full recompute):
# @dlt.table
# def gold_hourly_avg():
#     return (
#         dlt.read("silver_readings")
#         .groupBy(window("reading_timestamp", "1 hour"), "sensor_type")
#         .agg(avg("value").alias("avg_value"))
#     )

# RUNNABLE: Demonstrate the difference in behavior
from pyspark.sql.functions import *

readings = spark.table("iot_manufacturing_dw.sensor_readings")

# Streaming table equivalent: just append raw data
print("Streaming Table pattern: append raw data as-is")
print(f"  Would append {readings.count():,} rows incrementally")

# Materialized view equivalent: full recompute of aggregate
hourly_avg = (readings
    .withColumn("hour", date_trunc("hour", "reading_timestamp"))
    .groupBy("hour", "unit")
    .agg(
        count("*").alias("reading_count"),
        round(avg("value"), 4).alias("avg_value"),
        round(min("value"), 4).alias("min_value"),
        round(max("value"), 4).alias("max_value")
    )
    .orderBy("hour")
)
print(f"\nMaterialized View pattern: recompute aggregate")
print(f"  Produces {hourly_avg.count():,} aggregated rows")
hourly_avg.show(5)

## Streaming Tables vs Materialized Views — When to Use Each

### Decision Guide

```
Is your source append-only (new records only)?
├── YES → Use STREAMING TABLE
│   └── Examples: Kafka, Auto Loader, event logs
│   └── Processes only NEW data each run
│   └── Maintains offset/checkpoint
│
└── NO → Use MATERIALIZED VIEW
    └── Examples: Aggregations, joins, lookups
    └── Recomputes when upstream changes
    └── Handles updates and deletes

Special case: APPLY CHANGES INTO (SCD Type 1 or 2)
    └── Source has inserts + updates + deletes
    └── Use for CDC streams
```

### APPLY CHANGES INTO — CDC in DLT

```python
# DLT handles SCD Type 1 (overwrite) automatically:
dlt.apply_changes(
    target="silver_customers",
    source="bronze_customer_cdc",
    keys=["customer_id"],
    sequence_by="change_timestamp",
    apply_as_deletes=expr("operation = 'DELETE'"),
    except_column_list=["operation", "change_timestamp"]
)

# For SCD Type 2 (keep history):
dlt.apply_changes(
    target="dim_customers_history",
    source="bronze_customer_cdc",
    keys=["customer_id"],
    sequence_by="change_timestamp",
    stored_as_scd_type=2  # ← This is all you need!
)
```

> 💡 **DLT + SCD Type 2 = 3 lines of code** vs 50+ lines of manual MERGE logic!

In [ ]:
# Practice: Design a DLT pipeline
print("🎯 DLT PIPELINE DESIGN EXERCISE")
print("=" * 60)

# Scenario: E-commerce platform needs a Medallion pipeline
# Sources:
#   - S3 bucket: new order JSON files arrive every 5 minutes
#   - Kafka: real-time payment events
#   - PostgreSQL (via CDC): customer updates

# Design the DLT pipeline structure
pipeline_design = {
    "Bronze Layer": {
        "bronze_orders": {
            "type": "Streaming Table",
            "source": "Auto Loader (S3 JSON files)",
            "trigger": "Every 5 minutes",
            "expectations": ["order_id not null (warn)"],
        },
        "bronze_payments": {
            "type": "Streaming Table",
            "source": "Kafka topic: payments",
            "trigger": "Continuous",
            "expectations": ["payment_id not null (warn)"],
        },
        "bronze_customer_cdc": {
            "type": "Streaming Table",
            "source": "Kafka CDC from PostgreSQL",
            "trigger": "Continuous",
            "expectations": ["customer_id not null (warn)"],
        },
    },
    "Silver Layer": {
        "silver_orders": {
            "type": "Streaming Table",
            "source": "bronze_orders",
            "expectations": [
                "valid_order_id (drop)",
                "positive_amount (drop)",
                "valid_status (warn)",
            ],
        },
        "silver_payments": {
            "type": "Streaming Table",
            "source": "bronze_payments",
            "expectations": ["valid_payment_id (drop)", "positive_amount (drop)"],
        },
        "silver_customers": {
            "type": "APPLY CHANGES (SCD Type 2)",
            "source": "bronze_customer_cdc",
            "key": "customer_id",
            "sequence_by": "change_timestamp",
        },
    },
    "Gold Layer": {
        "gold_daily_revenue": {
            "type": "Materialized View",
            "source": "silver_orders",
            "refresh": "On upstream change",
        },
        "gold_customer_360": {
            "type": "Materialized View",
            "source": "silver_orders + silver_customers",
            "refresh": "On upstream change",
        },
        "gold_payment_reconciliation": {
            "type": "Materialized View",
            "source": "silver_orders + silver_payments",
            "refresh": "On upstream change",
        },
    }
}

for layer, tables in pipeline_design.items():
    print(f"\n🏗️ {layer}:")
    for table_name, config in tables.items():
        print(f"   📌 {table_name}")
        for key, value in config.items():
            if isinstance(value, list):
                print(f"      {key}:")
                for v in value:
                    print(f"         • {v}")
            else:
                print(f"      {key}: {value}")


---
# 📗 Section 6: Building a DLT Pipeline

## 6.1 DLT Version (Conceptual)

```python
# This entire notebook would be a DLT pipeline definition:

import dlt
from pyspark.sql.functions import *

@dlt.table(comment="Raw sensor readings")
def bronze_readings():
    return spark.table("iot_manufacturing_dw.sensor_readings")

@dlt.table(comment="Validated sensor readings")
@dlt.expect_or_drop("valid_value", "value IS NOT NULL AND value >= 0")
@dlt.expect_or_drop("valid_sensor", "sensor_id IS NOT NULL")
@dlt.expect("valid_quality", "quality_flag = 'valid'")
def silver_readings():
    return (dlt.read("bronze_readings")
        .filter("quality_flag = 'valid'")
        .withColumn("reading_hour", date_trunc("hour", "reading_timestamp"))
        .withColumn("_processed_at", current_timestamp())
    )

@dlt.table(comment="Hourly sensor aggregates")
def gold_hourly_metrics():
    return (dlt.read("silver_readings")
        .groupBy("reading_hour", "sensor_id")
        .agg(avg("value").alias("avg_value"),
             min("value").alias("min_value"),
             max("value").alias("max_value"),
             count("*").alias("reading_count"))
    )
```

## 6.2 Runnable Equivalent (Community Edition)

In [ ]:
# RUNNABLE: Bronze layer
from pyspark.sql.functions import *

spark.sql("USE iot_manufacturing_dw")

# Bronze: raw ingestion with audit
bronze_readings = (spark.table("sensor_readings")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source", lit("iot_gateway"))
)
bronze_readings.write.format("delta").mode("overwrite").saveAsTable("bronze_readings")
print(f"✅ Bronze readings: {spark.table('bronze_readings').count():,}")

In [ ]:
# RUNNABLE: Silver layer (with expectation-style quality checks)
from pyspark.sql.functions import *

bronze = spark.table("bronze_readings")
total = bronze.count()

# Apply expectations
silver_readings = (bronze
    .filter("value IS NOT NULL AND value >= 0")      # expect_or_drop
    .filter("sensor_id IS NOT NULL")                  # expect_or_drop
    .filter("quality_flag = 'valid'")                 # business filter
    .withColumn("reading_hour", date_trunc("hour", "reading_timestamp"))
    .withColumn("_processed_at", current_timestamp())
)

silver_readings.write.format("delta").mode("overwrite").saveAsTable("silver_readings")
silver_count = spark.table("silver_readings").count()
print(f"✅ Silver readings: {silver_count:,} (dropped {total - silver_count:,} invalid)")

In [ ]:
# RUNNABLE: Gold layer (hourly aggregates)
from pyspark.sql.functions import *

silver = spark.table("silver_readings")

gold_hourly = (silver
    .groupBy("reading_hour", "sensor_id")
    .agg(
        round(avg("value"), 4).alias("avg_value"),
        round(min("value"), 4).alias("min_value"),
        round(max("value"), 4).alias("max_value"),
        round(stddev("value"), 4).alias("stddev_value"),
        count("*").alias("reading_count")
    )
    .withColumn("_computed_at", current_timestamp())
)

gold_hourly.write.format("delta").mode("overwrite").saveAsTable("gold_hourly_metrics")
print(f"✅ Gold hourly metrics: {spark.table('gold_hourly_metrics').count():,}")

In [ ]:
# ============================================
# 🎯 YOUR TURN: Implement Silver Layer for Production Events
# ============================================
# Build a silver layer for production_events:
# 1. Read from production_events table
# 2. Filter: event_id NOT NULL, machine_id NOT NULL
# 3. Standardize: lowercase event_type
# 4. Add: event_hour (truncated timestamp), event_date
# 5. Add: _processed_at timestamp
# 6. Write to silver_production_events
# 7. Print counts
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
from pyspark.sql.functions import *

events = spark.table("production_events")
silver_events = (events
    .filter("event_id IS NOT NULL AND machine_id IS NOT NULL")
    .withColumn("event_type", lower(trim(col("event_type"))))
    .withColumn("event_hour", date_trunc("hour", "event_timestamp"))
    .withColumn("event_date", to_date("event_timestamp"))
    .withColumn("_processed_at", current_timestamp())
)
silver_events.write.format("delta").mode("overwrite").saveAsTable("silver_production_events")
print(f"✅ Silver production events: {spark.table('silver_production_events').count():,}")

---
# 📗 Section 7: Change Data Feed (CDF)

**What:** CDF records row-level changes (inserts, updates, deletes) to a Delta table.
Downstream consumers can read only the changes instead of the full table.

**Enable:** `ALTER TABLE ... SET TBLPROPERTIES (delta.enableChangeDataFeed = true)`

In [ ]:
%sql
-- Enable Change Data Feed on silver_readings
ALTER TABLE silver_readings SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Make a change to generate CDF records
UPDATE silver_readings SET value = value * 1.01 WHERE sensor_id = 1 AND reading_hour = (
  SELECT MIN(reading_hour) FROM silver_readings WHERE sensor_id = 1
);

In [ ]:
%sql
-- Read the change data feed
SELECT * FROM table_changes('silver_readings', 1)
LIMIT 20;

In [ ]:
%sql
-- CDF columns explained:
-- _change_type: 'insert', 'update_preimage', 'update_postimage', 'delete'
-- _commit_version: Delta version number
-- _commit_timestamp: When the change happened

SELECT _change_type, COUNT(*) AS cnt
FROM table_changes('silver_readings', 1)
GROUP BY _change_type;

---
# 📗 Section 8: Pipeline Orchestration

## Pipeline Dependency Graph

```
bronze_readings ──▶ silver_readings ──▶ gold_hourly_metrics
                                    ──▶ gold_anomaly_report
bronze_events ────▶ silver_events ────▶ gold_machine_health
bronze_quality ───▶ silver_quality ───▶ gold_shift_report
```

## Development vs Production Modes

| Aspect | Development | Production |
|---|---|---|
| Data | Sample/subset | Full dataset |
| Errors | Fail fast, verbose logs | Retry, alert, continue |
| Schedule | Manual trigger | Cron/event-driven |
| Compute | Small cluster | Auto-scaling |
| Quality | Warn on violations | Fail on critical |

In [ ]:
# Pipeline orchestration pattern (runnable)
import time
from datetime import datetime

class DLTPipelineSimulator:
    """Simulates DLT pipeline execution order."""
    
    def __init__(self, name):
        self.name = name
        self.steps = []
        self.results = {}
    
    def add_step(self, name, func, depends_on=None):
        self.steps.append({"name": name, "func": func, "depends_on": depends_on or []})
    
    def run(self):
        print(f"{'='*50}")
        print(f"Pipeline: {self.name}")
        print(f"Started: {datetime.now().strftime('%H:%M:%S')}")
        print(f"{'='*50}")
        
        for step in self.steps:
            deps_met = all(d in self.results for d in step["depends_on"])
            if not deps_met:
                print(f"  ⏭️ Skipping {step['name']} (deps not met)")
                continue
            try:
                start = time.time()
                result = step["func"]()
                elapsed = time.time() - start
                self.results[step["name"]] = result
                print(f"  ✅ {step['name']}: {result} ({elapsed:.1f}s)")
            except Exception as e:
                print(f"  ❌ {step['name']}: FAILED - {e}")
        
        print(f"{'='*50}")
        print(f"Completed: {len(self.results)}/{len(self.steps)} steps")

# Define pipeline
pipeline = DLTPipelineSimulator("IoT Monitoring Pipeline")

pipeline.add_step("bronze_readings", 
    lambda: f"{spark.table('bronze_readings').count():,} rows")
pipeline.add_step("silver_readings",
    lambda: f"{spark.table('silver_readings').count():,} rows",
    depends_on=["bronze_readings"])
pipeline.add_step("gold_hourly",
    lambda: f"{spark.table('gold_hourly_metrics').count():,} rows",
    depends_on=["silver_readings"])

pipeline.run()

## DLT Pipeline Configuration

### Pipeline Settings (databricks.yml or UI)

```yaml
# In your DABs bundle (databricks.yml):
resources:
  pipelines:
    techmart_medallion:
      name: "TechMart Medallion Pipeline"
      
      # Compute
      serverless: true          # Use serverless (recommended)
      # OR specify cluster:
      # clusters:
      #   - label: default
      #     num_workers: 4
      
      # Libraries (your pipeline code)
      libraries:
        - notebook:
            path: ./src/pipelines/bronze_layer
        - notebook:
            path: ./src/pipelines/silver_layer
        - notebook:
            path: ./src/pipelines/gold_layer
      
      # Target catalog/schema
      catalog: prod
      target: medallion
      
      # Pipeline mode
      continuous: false         # Triggered (not always-on)
      
      # Development mode (for testing)
      development: false        # Set true for dev environment
      
      # Notifications
      notifications:
        - email_recipients: ["data-team@company.com"]
          alerts: ["on-update-failure", "on-flow-failure"]
```

### Pipeline Modes

| Mode | Behavior | Cost | Use For |
|------|----------|------|---------|
| **Triggered** | Runs when triggered, then stops | Lower | Batch pipelines (hourly/daily) |
| **Continuous** | Always running, processes as data arrives | Higher | Real-time streaming |
| **Development** | Runs with smaller cluster, no optimization | Lowest | Testing and debugging |

### Monitoring DLT Pipelines

```python
# Query pipeline event log (system table)
SELECT
    timestamp,
    event_type,
    message,
    details
FROM event_log("pipeline_id")
WHERE event_type IN ('update_progress', 'flow_progress', 'error')
ORDER BY timestamp DESC
LIMIT 100

# Check expectation violations
SELECT
    flow_name,
    expectations.name,
    expectations.passed_records,
    expectations.failed_records
FROM event_log("pipeline_id")
WHERE event_type = 'flow_progress'
  AND details:flow_progress:metrics IS NOT NULL
```

In [ ]:
# ============================================
# 🎯 YOUR TURN: Write a DLT Pipeline
# ============================================
# Write a DLT pipeline for the techmart_dw database that:
#
# 1. Bronze: Read orders from a streaming source (simulate with readStream)
# 2. Silver: Clean orders with expectations:
#    - order_id NOT NULL (drop)
#    - total_amount > 0 (drop)
#    - status in ('completed','shipped','pending','cancelled') (warn)
# 3. Gold: Daily revenue aggregation
#
# Note: DLT code runs in a pipeline context, not a regular notebook.
# Write the code as if it's in a DLT notebook (import dlt, use decorators).


In [ ]:
# ============================================
# ✅ SOLUTION: DLT Pipeline Code
# ============================================
# This code would run inside a Databricks DLT pipeline notebook.
# In a regular notebook, we show the pattern:

dlt_pipeline_code = """
import dlt
from pyspark.sql.functions import col, lower, trim, sum, count, current_timestamp, lit

# ─── BRONZE ───────────────────────────────────────────────────────
@dlt.table(
    name="bronze_orders",
    comment="Raw orders — append-only, no transformations",
    table_properties={"quality": "bronze"}
)
def bronze_orders():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", "/checkpoints/orders/schema")
        .load("/raw/orders/")
        .withColumn("_ingested_at", current_timestamp())
    )

# ─── SILVER ───────────────────────────────────────────────────────
@dlt.table(
    name="silver_orders",
    comment="Validated and cleaned orders",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_drop("valid_order_id", "order_id IS NOT NULL")
@dlt.expect_or_drop("positive_amount", "total_amount > 0")
@dlt.expect("valid_status",
    "status IN ('completed', 'shipped', 'pending', 'cancelled')")
def silver_orders():
    return (
        dlt.read_stream("bronze_orders")
        .withColumn("status", lower(trim(col("status"))))
        .withColumn("total_amount", col("total_amount").cast("decimal(12,2)"))
        .filter(col("customer_id").isNotNull())
    )

# ─── GOLD ─────────────────────────────────────────────────────────
@dlt.table(
    name="gold_daily_revenue",
    comment="Daily revenue for Finance dashboard",
    table_properties={"quality": "gold"}
)
def gold_daily_revenue():
    return (
        dlt.read("silver_orders")
        .filter(col("status").isin("completed", "shipped"))
        .groupBy("order_date")
        .agg(
            count("*").alias("total_orders"),
            sum("total_amount").alias("total_revenue")
        )
        .orderBy("order_date")
    )
"""

print("📋 DLT PIPELINE CODE:")
print(dlt_pipeline_code)

print("\n💡 Key Points:")
print("   1. @dlt.table declares a table (Bronze, Silver, or Gold)")
print("   2. @dlt.expect* adds data quality rules")
print("   3. dlt.read_stream() for streaming sources")
print("   4. dlt.read() for batch/materialized views")
print("   5. DLT auto-detects dependencies from dlt.read() calls")
print("   6. No need to manage checkpoints, retries, or ordering!")


---
# 🚀 Section 9: Mini Projects

## Project 1: IoT Anomaly Detection Pipeline

In [ ]:
# Full pipeline: sensor readings → anomaly detection → alerts
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Silver: validated readings with anomaly scoring
silver = spark.table("silver_readings")
sensors = spark.table("sensors")

# Calculate per-sensor statistics for anomaly detection
w = Window.partitionBy("sensor_id")
anomaly_scored = (silver
    .join(sensors.select("sensor_id", "sensor_type"), "sensor_id")
    .withColumn("sensor_avg", avg("value").over(w))
    .withColumn("sensor_stddev", stddev("value").over(w))
    .withColumn("z_score", 
        (col("value") - col("sensor_avg")) / col("sensor_stddev"))
    .withColumn("is_anomaly_detected", 
        when(abs(col("z_score")) > 3, True).otherwise(False))
)

# Gold: anomaly summary
gold_anomalies = (anomaly_scored
    .filter("is_anomaly_detected = true")
    .groupBy("sensor_id", "sensor_type", "reading_hour")
    .agg(
        count("*").alias("anomaly_count"),
        round(avg("value"), 2).alias("avg_anomaly_value"),
        round(max(abs(col("z_score"))), 2).alias("max_z_score")
    )
    .orderBy(col("anomaly_count").desc())
)

gold_anomalies.write.format("delta").mode("overwrite").saveAsTable("gold_anomaly_alerts")
print(f"✅ Anomaly alerts: {spark.table('gold_anomaly_alerts').count():,} alert records")
gold_anomalies.show(10)

## Project 2: Production Quality Pipeline

In [ ]:
# Quality checks → validated → shift-level reports
from pyspark.sql.functions import *

# Silver: validated quality checks
qc = spark.table("quality_checks")
silver_qc = (qc
    .filter("check_id IS NOT NULL AND measurement IS NOT NULL")
    .withColumn("in_tolerance", 
        when(col("measurement").between(col("tolerance_min"), col("tolerance_max")), True)
        .otherwise(False))
    .withColumn("check_date", to_date("check_timestamp"))
    .withColumn("check_hour", hour("check_timestamp"))
    .withColumn("shift",
        when(col("check_hour").between(6, 13), "morning")
        .when(col("check_hour").between(14, 21), "afternoon")
        .otherwise("night"))
    .withColumn("_processed_at", current_timestamp())
)
silver_qc.write.format("delta").mode("overwrite").saveAsTable("silver_quality_checks")

# Gold: shift-level quality report
gold_shift = (silver_qc
    .groupBy("check_date", "shift", "machine_id")
    .agg(
        count("*").alias("total_checks"),
        sum(when(col("in_tolerance"), 1).otherwise(0)).alias("passed"),
        sum(when(~col("in_tolerance"), 1).otherwise(0)).alias("failed"),
        round(avg("measurement"), 4).alias("avg_measurement")
    )
    .withColumn("pass_rate", round(col("passed") / col("total_checks") * 100, 2))
    .withColumn("_computed_at", current_timestamp())
)

gold_shift.write.format("delta").mode("overwrite").saveAsTable("gold_shift_quality")
print(f"✅ Shift quality report: {spark.table('gold_shift_quality').count():,} rows")

# Show worst performing shifts
gold_shift.orderBy("pass_rate").show(10)

---
# 🏆 Section 10: Interview Questions

## Q1: What is DLT and how does it differ from regular Spark jobs?

**Answer:** DLT (now Lakeflow Spark Declarative Pipelines) is a declarative framework where you define
*what* your data should look like, not *how* to process it. The engine handles execution order,
incremental processing, retries, and schema management. Regular Spark jobs are imperative —
you control every step manually.

## Q2: Explain expectations and the three violation modes

**Answer:**
- `expect` — logs a warning metric but keeps the row (monitoring)
- `expect_or_drop` — silently removes rows that violate (filtering)
- `expect_or_fail` — stops the entire pipeline (critical integrity)

Use `expect` for monitoring, `expect_or_drop` for non-critical cleaning,
`expect_or_fail` for data that must never be wrong (PKs, critical business rules).

## Q3: What is Auto Loader and when would you use it?

**Answer:** Auto Loader (`cloudFiles`) incrementally ingests new files from cloud storage.
It uses a checkpoint to track processed files, ensuring exactly-once semantics.
Use it when: data arrives as files (JSON, CSV, Parquet), you need incremental processing,
and you want automatic schema inference/evolution.

## Q4: Streaming tables vs materialized views?

**Answer:**
- **Streaming tables:** Append-only, process only new data. Use for Bronze ingestion, logs, events.
- **Materialized views:** Full recompute, support updates/deletes. Use for Silver/Gold aggregations.

Key difference: streaming tables can't handle source updates; materialized views can.

## Q5: How does Change Data Feed work?

**Answer:** CDF records row-level changes (insert, update_preimage, update_postimage, delete)
to a Delta table. Enable with `delta.enableChangeDataFeed = true`. Downstream consumers
read only changes since a version/timestamp using `table_changes()`, avoiding full table scans.
Use for: incremental downstream updates, audit trails, real-time sync.

---
# ✅ Notebook Complete!

**What was covered:**
- DLT/SDP concepts: declarative pipelines, lifecycle, when to use
- Syntax: `@dlt.table`, `@dlt.view` (legacy) and `@pipelines.table` (new 2025)
- Expectations: three violation modes with runnable equivalents
- Auto Loader: incremental file ingestion concepts
- Streaming tables vs materialized views
- Full Bronze → Silver → Gold pipeline (runnable on Community Edition)
- Change Data Feed: enabling and reading row-level changes
- Pipeline orchestration patterns
- 2 mini projects: IoT Anomaly Detection, Production Quality
- 5 interview questions

**Database created:** `iot_manufacturing_dw` (5 tables, 270K+ rows)

**Next:** Notebook 07 — Performance Optimization

---
*All databases remain available for subsequent notebooks.*